In [1]:
%%capture
!pip install pydub
!pip install noisereduce
!pip install json-tricks
!pip install tensorflow
!pip install scikit-learn
!pip install keras
!pip install librosa
!pip install seaborn

In [3]:
%%capture
import numpy as np
import pandas as pd
import os
from json_tricks import dump, load

from pydub import AudioSegment, effects
import librosa
import noisereduce as nr

import tensorflow as tf
import keras
import sklearn

import time


from keras.models import Sequential
from keras import layers
from keras import optimizers
from keras import callbacks
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

In [4]:
def extract_features(data, sample_rate):
    result = np.array([])

    # Ensure n_fft is never larger than the audio signal length
    n_fft = min(512, len(data))
    hop_length = n_fft // 2

    # Zero-Crossing Rate (ZCR)
    zcr = np.mean(librosa.feature.zero_crossing_rate(y=data, hop_length=hop_length).T, axis=0)
    result = np.hstack((result, zcr))  # stacking horizontally

    # Chroma_stft
    stft = np.abs(librosa.stft(data, n_fft=n_fft, hop_length=hop_length))
    chroma_stft = np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate, n_fft=n_fft, hop_length=hop_length).T, axis=0)
    result = np.hstack((result, chroma_stft))  # stacking horizontally

    # MFCC
    mfcc = np.mean(librosa.feature.mfcc(y=data, sr=sample_rate, n_fft=n_fft, hop_length=hop_length).T, axis=0)
    result = np.hstack((result, mfcc))  # stacking horizontally

    # Root Mean Square Value (RMS)
    rms = np.mean(librosa.feature.rms(y=data, hop_length=hop_length).T, axis=0)
    result = np.hstack((result, rms))  # stacking horizontally

    # MelSpectrogram
    mel = np.mean(librosa.feature.melspectrogram(y=data, sr=sample_rate, n_fft=n_fft, hop_length=hop_length).T, axis=0)
    result = np.hstack((result, mel))  # stacking horizontally

    # Spectral Contrast
    spectral_contrast = np.mean(librosa.feature.spectral_contrast(S=stft, sr=sample_rate, n_fft=n_fft, hop_length=hop_length).T, axis=0)
    result = np.hstack((result, spectral_contrast))  # stacking horizontally

    # Spectral Rolloff
    spectral_rolloff = np.mean(librosa.feature.spectral_rolloff(y=data, sr=sample_rate, n_fft=n_fft, hop_length=hop_length).T, axis=0)
    result = np.hstack((result, spectral_rolloff))  # stacking horizontally

    # Spectral Bandwidth
    spectral_bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=data, sr=sample_rate, n_fft=n_fft, hop_length=hop_length).T, axis=0)
    result = np.hstack((result, spectral_bandwidth))  # stacking horizontally

    # Tonnetz (Tonal Centroids)
    harmonic = librosa.effects.harmonic(y=data, n_fft=n_fft, hop_length=hop_length)
    tonnetz = np.mean(librosa.feature.tonnetz(y=harmonic, hop_length=hop_length, sr=sample_rate).T, axis=0)
    result = np.hstack((result, tonnetz))  # stacking horizontally

    return result

In [6]:
# Reading the model from JSON file

saved_model_path = 'model 81.json'
saved_weights_path = 'model 81.weights.h5'

with open(saved_model_path , 'r') as json_file:
    json_savedModel = json_file.read()

# Loading the model architecture, weights
model = tf.keras.models.model_from_json(json_savedModel)
model.load_weights(saved_weights_path)

# Compiling the model with similar parameters as the original model.
model.compile(loss='categorical_crossentropy',
                optimizer=tf.keras.optimizers.AdamW(learning_rate=0.001),
                metrics=['categorical_accuracy'])

# Model's structure visualization
# tf.keras.utils.plot_model(model, to_file='model.png', show_shapes=True, show_layer_names=True)

In [7]:
import joblib
scaler = joblib.load('scaler 81.pkl')

In [8]:
def processData(data, sample_rate):
    # data, sample_rate = librosa.load(path)
    data = nr.reduce_noise(y=data, sr=sample_rate)
    data = librosa.util.normalize(data)
    data, index = librosa.effects.trim(data, top_db=20)
    
    features = extract_features(data, sample_rate)
    return features

In [9]:
def getEmo(path):
    data, sample_rate = librosa.load(path)
    audio = processData(data, sample_rate)
    audio = scaler.transform(audio.reshape(1, -1))
    audio = np.expand_dims(audio, axis = 1 )
    return model(audio)

In [10]:
emo = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']

In [28]:
res = getEmo('../Records/Recording (7).wav')
res

<tf.Tensor: shape=(1, 8), dtype=float32, numpy=
array([[4.8200767e-03, 9.8015962e-04, 6.4172423e-03, 1.7096399e-04,
        9.7784847e-01, 1.6631564e-03, 5.1186779e-03, 2.9812327e-03]],
      dtype=float32)>

In [29]:
emo[np.argmax(res,axis = 1)[0]]

'angry'